In [ ]:
import sympy as sp
import numpy as np
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display, clear_output

# ==============================================================================
# PROBLEM 2: Symbolic Z-Transform and Interactive Pole-Zero / ROC Visualization
# Signal: x[n] = (alpha^n + alpha^-n) * u[n]
# ==============================================================================

explanation_text = """
<div style="background-color: #f8f9fa; padding: 10px; border-radius: 5px; border: 1px solid #dee2e6; font-size: 13px;">
<b>Solution Overview</b><br>
* <b>Signal:</b> x[n] = (α^n + α^-n)u[n]<br>
* <b>Z-Transform Derivation:</b> X(z) = z(2αz - α^2 - 1) / ((z - α)(αz - 1))<br>
* <b>Poles & Zeros:</b> Zeros at z = 0 and z = (α^2 + 1)/(2α), Poles at p₁ = α and p₂ = 1/α.<br>
* <b>ROC (Region of Convergence):</b> |z| > max(|α|, 1/|α|).<br>
* <b>Note:</b> Use the slider below to dynamically change the parameter α and observe the movement of poles, zeros, and ROC boundary.
</div>
"""
display(widgets.HTML(explanation_text))

out = widgets.Output()

n_samples = 30
n_vec = np.arange(n_samples)

def plot_problem_2(alpha_val):
    with out:
        clear_output(wait=True)
        
        fig, (ax_pz, ax_time) = plt.subplots(1, 2, figsize=(16, 5), gridspec_kw={'width_ratios': [1, 2]})
        plt.subplots_adjust(wspace=0.25)

        # --- 1. Pole-Zero Map & ROC ---
        ax_pz.set_aspect('equal')
        ax_pz.set_xlim(-3.0, 3.0)
        ax_pz.set_ylim(-3.0, 3.0)
        ax_pz.axhline(0, color='black', linewidth=1)
        ax_pz.axvline(0, color='black', linewidth=1)
        ax_pz.grid(True, linestyle=':', alpha=0.7)

        # Poles and radii calculation matching the textbook diagram
        p1 = alpha_val
        p2 = 1.0 / alpha_val if alpha_val != 0 else np.sign(alpha_val)*np.inf
        
        r_inner = abs(alpha_val)
        r_outer = abs(1.0 / alpha_val) if alpha_val != 0 else 3.0
        roc_radius = max(r_inner, r_outer)

        # Draw both concentric circles corresponding to the two poles' radii
        theta = np.linspace(0, 2*np.pi, 200)
        ax_pz.plot(r_inner * np.cos(theta), r_inner * np.sin(theta), 'k-', alpha=0.7, label=f'r = |α| ({r_inner:.2f})')
        ax_pz.plot(r_outer * np.cos(theta), r_outer * np.sin(theta), 'k-', alpha=0.7, label=f'r = 1/|α| ({r_outer:.2f})')

        # Region of Convergence (ROC) shading for |z| > roc_radius
        r_roc = np.linspace(roc_radius, 4.0, 50)
        R, T = np.meshgrid(r_roc, theta)
        X_roc = R * np.cos(T)
        Y_roc = R * np.sin(T)
        ax_pz.contourf(X_roc, Y_roc, R, levels=20, cmap='Greens', alpha=0.2)

        # Unit Circle reference
        ax_pz.plot(np.cos(theta), np.sin(theta), 'k--', alpha=0.4, label='Unit Circle')

        # Zeros: z1 = 0, z2 = (alpha^2 + 1) / (2*alpha)
        z2 = (alpha_val**2 + 1) / (2 * alpha_val) if alpha_val != 0 else 0
        ax_pz.scatter([0, z2], [0, 0], s=120, facecolors='none', edgecolors='b', linewidths=2, marker='o')

        # Poles: p1 = alpha, p2 = 1/alpha (using magenta/purple crosses as shown in the textbook figure)
        ax_pz.scatter([p1], [0], s=140, color='purple', marker='x', linewidths=3, label=f'Pole p₁ = α')
        if abs(p1 - p2) > 1e-4:
            ax_pz.scatter([p2], [0], s=140, color='purple', marker='x', linewidths=3, label=f'Pole p₂ = 1/α')

        stability = "Stable" if roc_radius < 1 else "Unstable"
        ax_pz.set_title(f'Pole-Zero Map & ROC ({stability}, ROC: |z| > {roc_radius:.2f})', fontsize=10, fontweight='bold')
        ax_pz.set_xlabel('Real Part', fontsize=9)
        ax_pz.set_ylabel('Imaginary Part', fontsize=9)

        # Legend handles in a single straight line
        circle_handle = plt.Line2D([0], [0], color='k', linestyle='-', alpha=0.7, label='Concentric Circles (r = |α|, 1/|α|)')
        pole_handle = plt.Line2D([0], [0], marker='x', color='purple', markersize=8, markeredgewidth=3, linestyle='None', label='Poles (p₁, p₂)')
        zero_handle = plt.Line2D([0], [0], marker='o', markerfacecolor='none', markeredgecolor='b', markersize=8, markeredgewidth=2, linestyle='None', label='Zeros (z=0, z₂)')

        ax_pz.legend(handles=[circle_handle, pole_handle, zero_handle], loc='upper center', bbox_to_anchor=(0.5, -0.15), ncol=3, fontsize=8)

        # --- 2. Time Domain Plot ---
        if abs(alpha_val) < 1e-3:
            x_n_vals = np.zeros(n_samples)
        else:
            x_n_vals = (alpha_val**n_vec) + (alpha_val**(-n_vec))

        ax_time.stem(n_vec, x_n_vals, linefmt='r-', markerfmt='ro', basefmt='k-')
        ax_time.set_title(f'Temporal Evolution: x[n] = (α^n + α^-n)u[n] for α={alpha_val:.2f}', fontsize=10, fontweight='bold')
        ax_time.set_xlabel('Time index n', fontsize=9)
        ax_time.set_ylabel('x[n]', fontsize=9)
        ax_time.set_xlim(-1, n_samples)
        
        max_abs_val = np.max(np.abs(x_n_vals))
        y_limit = max(2.0, min(max_abs_val * 1.25, 50.0))
        ax_time.set_ylim(-y_limit, y_limit)
        ax_time.grid(True, linestyle=':', alpha=0.7)

        plt.show()

# Slider setup for alpha parameter (restricted within (0, 1) to match textbook figure (b))
alpha_slider = widgets.FloatSlider(value=0.5, min=0.1, max=0.9, step=0.01, description='Alpha:', style={'description_width': 'initial'})

plot_problem_2(alpha_slider.value)

interactive_plot = widgets.interactive(plot_problem_2, alpha_val=alpha_slider)
display(widgets.VBox([interactive_plot, out]))